# Assignment 1 – EDA & Baselines
**Competition:** [Hull Tactical – Market Prediction](https://www.kaggle.com/competitions/hull-tactical-market-prediction)

Goal: each day, choose how much of the S&P 500 to hold (**position between 0 and 2**) so that the
volatility‑adjusted Sharpe ratio is as high as possible.

Contents
1. Setup (Colab)
2. Data overview
3. Missing values
4. Target analysis
5. Feature–target relationships
6. Baselines *(added in the next step)*

## 1. Setup
Run this on Colab. Upload your `kaggle.json` (Kaggle → Settings → *Create New Token*) when asked.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = 'https://github.com/Kutay33/GROUP60_CENG313.git'   # TODO: your repo URL

if IN_COLAB:
    if not os.path.exists('GROUP60_CENG313'):
        !git clone -q $REPO
    %cd GROUP60_CENG313
    if not os.path.exists('data/train.csv'):
        from google.colab import files
        files.upload()                      # select kaggle.json
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        !kaggle competitions download -c hull-tactical-market-prediction -p data -q
        !unzip -oq data/hull-tactical-market-prediction.zip -d data
sys.path.append('.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data as D
from src import utils as U

U.set_seed(42)
pd.set_option('display.max_columns', 50)

## 2. Data overview

In [ ]:
train = D.load_train()
test = D.load_test()
feats = D.feature_columns(train)
print('train:', train.shape, ' test:', test.shape, ' features:', len(feats))
train.head()

In [ ]:
groups = pd.Series([D.group_of(c) for c in feats]).value_counts().sort_index()
groups.index = [f"{g} – {D.FEATURE_GROUPS[g]}" for g in groups.index]
groups.to_frame('n_features')

In [ ]:
print('Columns only in train:', sorted(set(train.columns) - set(test.columns)))
print('Columns only in test :', sorted(set(test.columns) - set(train.columns)))

**Note:** `forward_returns`, `risk_free_rate` and `market_forward_excess_returns` describe the *next* day,
so they are targets / scoring inputs, not features. The test set only has their **lagged** versions.

## 3. Missing values

In [ ]:
miss = D.missing_summary(train)
miss.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
avail = train[feats].notna().mean(axis=1).rolling(50).mean()
ax.plot(train['date_id'], avail)
ax.set(title='Share of features available per day (50-day rolling)', xlabel='date_id', ylabel='share')
plt.show()

In [ ]:
miss.groupby('group')['missing_ratio'].mean().sort_values().plot.barh(figsize=(6, 3), title='Mean missing ratio by group')
plt.show()

Older dates have many missing features; most features become available later. In the baselines we
drop the oldest, mostly empty rows and forward‑fill the rest.

## 4. Target analysis

In [ ]:
y = train[D.TARGET]
y.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(y, bins=100); axes[0].set_title('Excess return distribution')
axes[1].plot(train['date_id'], (1 + train['forward_returns']).cumprod()); axes[1].set_yscale('log')
axes[1].set_title('S&P 500 wealth (log)')
axes[2].plot(train['date_id'], train['forward_returns'].rolling(252).std() * np.sqrt(252))
axes[2].set_title('Rolling 1y volatility')
plt.tight_layout(); plt.show()
print('skew:', y.skew().round(3), ' kurtosis:', y.kurt().round(3), ' share positive days:', (y > 0).mean().round(3))

In [ ]:
from pandas.plotting import autocorrelation_plot
fig, ax = plt.subplots(figsize=(10, 3))
lags = range(1, 21)
ax.bar(lags, [y.autocorr(l) for l in lags], label='returns')
ax.plot(lags, [y.abs().autocorr(l) for l in lags], 'r.-', label='|returns| (volatility clustering)')
ax.axhline(0, color='k', lw=.5); ax.legend(); ax.set_title('Autocorrelation')
plt.show()

Returns are close to unpredictable (tiny autocorrelation) but **absolute** returns are strongly
autocorrelated → volatility is predictable. This suggests a strategy that lowers exposure in high‑volatility periods.

## 5. Feature–target relationships

In [ ]:
tr = D.drop_early_rows(train)
corr = tr[feats].corrwith(tr[D.TARGET]).dropna().sort_values()
fig, ax = plt.subplots(figsize=(12, 4))
corr.plot.bar(ax=ax, color=['#d1242f' if v < 0 else '#1f6feb' for v in corr])
ax.set_title('Correlation of each feature with next-day excess return'); ax.tick_params(axis='x', labelsize=6)
plt.show()
pd.concat([corr.head(8), corr.tail(8)]).to_frame('corr')

In [ ]:
# Is the correlation stable over time? Compare the first and second half.
half = len(tr) // 2
c1 = tr.iloc[:half][feats].corrwith(tr.iloc[:half][D.TARGET])
c2 = tr.iloc[half:][feats].corrwith(tr.iloc[half:][D.TARGET])
plt.figure(figsize=(5, 5)); plt.scatter(c1, c2, s=10)
plt.axhline(0, c='k', lw=.5); plt.axvline(0, c='k', lw=.5)
plt.xlabel('corr – first half'); plt.ylabel('corr – second half'); plt.title('Correlation stability')
plt.show()
print('rank correlation between halves:', round(c1.corr(c2, method='spearman'), 3))

In [ ]:
top = corr.abs().sort_values(ascending=False).index[:15]
plt.figure(figsize=(8, 6))
plt.imshow(tr[top].corr(), cmap='RdBu_r', vmin=-1, vmax=1)
plt.xticks(range(len(top)), top, rotation=90); plt.yticks(range(len(top)), top)
plt.colorbar(); plt.title('Correlation among top-15 features'); plt.show()

### EDA takeaways (fill in with your own observations)
- Signal is weak: |correlations| are small and not very stable over time → strong regularisation, time‑based CV.
- Volatility clusters → position sizing by predicted volatility is promising.
- Many early rows are missing → drop or impute carefully, never fill with future values.